### TRABAJO PRÁCTICO N° 4

##### MÉTODOS SUPERVISADOS: REGRESIÓN & CLASIFICACIÓN USANDO LA EHP

#### A. Enfoque de validación

##### Punto 1

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm    

# ─────────────────────── 1. FUNCIÓN DE MAPE0 ───────────────────────────────
def unificar_estado(valor):
    if pd.isna(valor):
        return np.nan

    if isinstance(valor, str):
        valor = valor.strip().lower()
        if "ocupado" in valor and "des" not in valor:
            return 1
        elif "desocupado" in valor:
            return 2
        elif "inactivo" in valor:
            return 3
        elif "menor de 10" in valor:
            return 4
        elif "entrevista individual no realizada" in valor:
            return 0
        else:
            return np.nan

    try:
        valor_int = int(valor)
    except (ValueError, TypeError):
        return np.nan

    mapa_num = {1: 1, 2: 2, 3: 3, 4: 4, 0: 0}
    return mapa_num.get(valor_int, np.nan)

# ─────────────────────── 2. CARGA Y PREPARACIÓN ────────────────────────────
def cargar_y_preparar_datos():
      # Cargar los datasets
    t2004 = pd.read_stata(
        "C:/Users/User/Downloads/Individual_t104.dta",
        convert_categoricals=True
    )
    t2024 = pd.read_excel(
        "C:/Users/User/Downloads/usu_individual_T124.xlsx"
    )

    # Columnas en mayúsculas
    for df in (t2004, t2024):
        df.columns = df.columns.str.upper()


    # Aplicar mapeo de estado
    t2004["ESTADO"] = t2004["ESTADO"].apply(unificar_estado)
    t2024["ESTADO"] = t2024["ESTADO"].apply(unificar_estado)

    # Filtrar por región
    t2004 = t2004[t2004["REGION"] == "Gran Buenos Aires"]
    t2024 = t2024[t2024["REGION"] == 1]

    # Conservar columnas comunes
    columnas_comunes = list(set(t2004.columns) & set(t2024.columns))
    t2004 = t2004[columnas_comunes].copy()
    t2024 = t2024[columnas_comunes].copy()

    # Añadir año
    t2004["ANO_EPH"] = 2004
    t2024["ANO_EPH"] = 2024

    # Concatenar
    df = pd.concat([t2004, t2024], ignore_index=True)

    # Crear variable desocupado solo en casos válidos
    respondieron = df[df["ESTADO"] != 0].copy()
    
    # Crear la variable binaria
    respondieron["desocupado"] = (respondieron["ESTADO"] == 2).astype(int)
    respondieron["ESTADO"] = respondieron["ESTADO"].astype(int)
    respondieron = respondieron.dropna(subset=["desocupado"])       

    norespondieron = df[df["ESTADO"] == 0].copy()

    return respondieron, norespondieron




# ─────────────────────── 3. TRANSFORMACIONES ───────────────────────────────
def procesar_datos(df, drop_empty=True):
    excluir = ['desocupado', 'ANO_EPH', 'CODUSU', 'ANO4', 'TRIMESTRE', 'NRO_HOGAR', 'COMPONENTE', 'REGION', 'ESTADO', 
               'PP02C8', 'PP02C4', 'PP02C7', 'PP02C1', 'PP02C5', 'PP02C3', 'PP02C6', 'PP02C2', 'PP11B1']
    categoricas = [
    # CH
    'CH03', 'CH04', 'CH07', 'CH08', 'CH09', 'CH10', 'CH11', 'CH12', 'CH13', 'CH15', 'CH16',
    # PP02
    'PP02C1', 'PP02C2', 'PP02C3', 'PP02C4', 'PP02C5', 'PP02C6', 'PP02C7', 'PP02C8',
    'PP02E', 'PP02H', 'PP02I',
    # PP03
    'ESTADO', 'CAT_OCUP', 'CAT_INAC',
    'PP03C', 'PP03D', # PP03D si tiene pocos valores únicos
    'PP03G', 'PP03H', 'PP03I', 'PP03J',
    # PP04
    'PP04A', 'PP04B_COD', 'PP04B1', 'PP04B2', 'PP04C', # PP04C si son rangos codificados
    'PP04D_COD', 'PP04G',
    # PP05
    'PP05C_1', 'PP05C_2', 'PP05C_3', 'PP05E', 'PP05F', 'PP05H',
    # PP06 (solo las que son códigos, NO montos)
    'PP06A', 'PP06E', 'PP06H',
    # PP07 (asalariados)
    'PP07A', 'PP07C', 'PP07D', 'PP07E',
    'PP07F1', 'PP07F2', 'PP07F3', 'PP07F4', 'PP07F5',
    'PP07G1', 'PP07G2', 'PP07G3', 'PP07G4',
    'PP07H', 'PP07I', 'PP07J', 'PP07K',
    # PP09
    'PP09A', 'PP09B', 'PP09C', # Y sus variantes ESP
    # PP10 y PP11 (revisar todas, muchas son códigos)
    'PP10A', 'PP10C', 'PP10D', 'PP10E',
    'PP11A', 'PP11B1', 'PP11B_COD', 'PP11C', 'PP11C99', 'PP11D_COD',
    'PP11L', 'PP11L1', 'PP11M', 'PP11N', 'PP11O', 'PP11P', 'PP11Q', 'PP11R', 'PP11S', 'PP11T',
    # Otras que tenías
    'REGION', 'NIVEL_ED', 'INTENSI', # NIVEL_ED es CH12 usualmente. INTENSI es categórica.
    'MAS_500', # Si la usas como feature
    # 'AGLOMERADO', # Si la usas como feature
    'PP04B1', 'PP05C_1', 'CH07', 'PP11M', 'PP05H', 'CH04', 'PP10A', 'PP06E', 'PP04G', 'CH15_COD', 'PP04A', 'PP09A_ESP', 'PP11B1', 'PP02C4', 'PP07G3', 'PP11D_COD', 'MAS_500', 'PP11R', 'PP03J', 'PP07G4', 'PP11P', 'CH16_COD', 'PP11N', 'PP05E', 'PP09A', 'CAT_OCUP', 'PP03I', 'PP05F', 'PP07G_59', 'INTENSI', 'CH09', 'PP04D_COD', 'PP06A', 'CH13', 'CH16', 'PP10C', 'PP02C7', 'PP11B_COD', 'CAT_INAC', 'PP05C_2', 'PP09C', 'PP09B', 'PP07F1', 'PP02C6', 'PP07A', 'PP02C5', 'PP09C_ESP', 'PP07G2', 'PP11A', 'PP07I', 'PP03C', 'PP05C_3', 'PP11T', 'PP03G', 'PP07C', 'CH08', 'PP11C99', 'PP10E', 'PP07E', 'CH12', 'PP10D', 'PP11C', 'PP07F4', 'PP02C1', 'PP06H', 'PP11L1', 'PP02H', 'CH11', 'REGION', 'PP11L', 'CH14', 'CH03', 'CH10', 'PP02C2', 'PP11O', 'NIVEL_ED', 'PP02C8', 'PP02E', 'PP07D', 'PP03H', 'PP11S', 'PP07H', 'PP02I', 'PP07F3', 'H15', 'PP11Q', 'PP04B_COD', 'PP07F5', 'PP07F2', 'CH15', 'PP04C99', 'PP04C', 'PP07G1', 'PP07J', 'PP07K', 'PP02C3', 'ESTADO', 'PP03D', 'PP04B2'
]



    return df

def filtrar_correlacion(X, threshold=0.95):
    """Elimina una de las variables en pares altamente correlacionados."""
    # Matriz de correlación absoluta
    corr_matrix = X.corr().abs()

    # Triangular superior
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Encontrar columnas con correlación > threshold
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

    print(f"Variables eliminadas por alta correlación (> {threshold}): {to_drop}")

    return X.drop(columns=to_drop)




def split_y_diferencias(df, año):
    df_año = df[df['ANO_EPH'] == año].copy()
    df_proc = procesar_datos(df_año)

    excluir = ['desocupado', 'ANO_EPH', 'CODUSU', 'ANO4', 'TRIMESTRE', 'NRO_HOGAR', 'COMPONENTE', 'REGION', 'ESTADO', 
               'PP02C8', 'PP02C4', 'PP02C7', 'PP02C1', 'PP02C5', 'PP02C3', 'PP02C6', 'PP02C2', 'PP11B1']
    X = df_proc.drop(excluir, axis=1)
    X = X.select_dtypes(include=['number'])

    # Filtrar columnas con varianza nula
    X = X.loc[:, X.var() > 0]

    y = df_proc['desocupado']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=444, stratify=y)

    # Comparación de estadísticas
    diff = pd.DataFrame({
        'Media Entrenamiento': X_train.mean(),
        'Desvío Entrenamiento': X_train.std(),
        'Media Prueba': X_test.mean(),
        'Desvío Prueba': X_test.std(),
        'Diferencia de Medias': X_train.mean() - X_test.mean()
    }).sort_values('Diferencia de Medias', key=abs, ascending=False)

    return X_train, X_test, y_train, y_test, diff



# ─────────────────────── 4. EJECUCIÓN ──────────────────────────────────────
respondieron, norespondieron = cargar_y_preparar_datos()

X_train_2004, X_test_2004, y_train_2004, y_test_2004, diff_2004 = split_y_diferencias(respondieron, 2004)
print("\nDiferencias 2004 (Top 20):")
print(diff_2004.head(20))

X_train_2024, X_test_2024, y_train_2024, y_test_2024, diff_2024 = split_y_diferencias(respondieron, 2024)
print("\nDiferencias 2024 (Top 20):")
print(diff_2024.head(20))



Diferencias 2004 (Top 20):
          Media Entrenamiento  Desvío Entrenamiento  Media Prueba  \
P21                267.523293            926.496545    254.121291   
P47T               366.472965            988.239959    354.852094   
PP08D1             199.036109            872.402291    191.838569   
ITF               1239.571375           1428.319106   1233.475131   
PP06D               21.547240            218.075086     16.590314   
IPCF               364.102677            841.492748    360.150463   
PONDERA           1625.573059            356.363766   1621.866928   
PP3E_TOT            18.260992             60.044272     15.201134   
PP08J1              12.207671             88.736188      9.761780   
V2_M                44.492610            191.954581     46.068499   
PP06C               39.649579            255.380368     38.175393   
V8_M                 4.836296             71.076361      3.647469   
V12_M                8.347428             88.998179      7.188045   
V9_M  

In [2]:
respondieron.groupby("ANO_EPH")["desocupado"].value_counts()

ANO_EPH  desocupado
2004     0             7109
         1              528
2024     0             6699
         1              311
Name: count, dtype: int64

#### B. Metodo Supervisado 1: Modelo de Regresión Lineal

##### Punto 2

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant

def preparar_datos_para_modelos(df):
    df = df.copy()
    
    df['edad'] = pd.to_numeric(df['CH06'], errors='coerce')
    df['edad2'] = df['edad']**2
    df['mujer'] = (df['CH04'] == 2).astype(int)
    
    def calcular_educ(row):
        valor = row['CH12']
        
        niveles_texto = {
            'jardín/preescolar': 0,
            'educación especial (discapacitado)': 0,
            'primario': 0,
            'egb': 0,
            'secundario': 7,
            'polimodal': 7,
            'terciario': 13,
            'universitario': 13,
            'posgrado universitario': 18,
        }
        
        niveles_num = {
            0: 0, 1: 0, 2: 3, 3: 7, 4: 10, 5: 12, 6: 14, 7: 16, 8: 16, 9: 17, 10: 19,
            99: np.nan
        }
        
        try:
            valor_num = float(valor)
            if valor_num in niveles_num:
                return niveles_num[valor_num]
        except:
            pass
        
        if isinstance(valor, str):
            v_lower = valor.strip().lower()
            if v_lower in niveles_texto:
                return niveles_texto[v_lower]
        
        return np.nan
    
    df['educ'] = df.apply(calcular_educ, axis=1)

    # Codificar variable tiene_cobertura desde CH14
    df['tiene_cobertura'] = (df['CH08'] == 1).astype(int)


    # Codificar variable sabe_leer_escribir desde CH09
    def codificar_lectoescritura(valor):
        if pd.isna(valor):
            return np.nan
        valor_str = str(valor).strip().lower()
        if valor_str in ['1', 'sí', 'si']:
            return 1
        elif valor_str in ['2', 'no']:
            return 0
        elif valor_str in ['3', 'menor de 2 años', '9', 'ns./nr.']:
            return np.nan
        return np.nan

    df['sabe_leer_escribir'] = df['CH09'].apply(codificar_lectoescritura)

    # Calcular salario semanal ajustado por IPC
    ipc_base = 100
    ipc_actual = 235.125
    factor_ajuste = ipc_actual / ipc_base
    df['salario_semanal'] = (pd.to_numeric(df['P21'], errors='coerce') * factor_ajuste) / (52 / 12)
    
    # Filtrar outliers y no válidos
    q_low = df['salario_semanal'].quantile(0.01)
    q_hi = df['salario_semanal'].quantile(0.99)
    df = df[(df['salario_semanal'] > 0) &
            (df['salario_semanal'] >= q_low) &
            (df['salario_semanal'] <= q_hi)].copy()
    
    df['log_salario'] = np.log(df['salario_semanal'])
    
    return df



def estimar_modelos(train_data):
    """Estima los modelos solicitados en la consigna 2 y devuelve los modelos entrenados"""
    # Configuración de modelos según consigna
    modelos_config = [
    ['edad'],
    ['edad', 'edad2'],
    ['edad', 'edad2', 'educ'],
    ['edad', 'edad2', 'educ', 'mujer'],
    ['edad', 'edad2', 'educ', 'mujer', 'tiene_cobertura', 'sabe_leer_escribir']
    ]

    resultados = []
    modelos_entrenados = []

    for i, vars_modelo in enumerate(modelos_config, 1):
        try:
            # Seleccionar datos y eliminar NA
            datos = train_data[vars_modelo + ['log_salario']].dropna()
            
            if len(datos) < 10:
                print(f"Modelo {i} omitido - solo {len(datos)} observaciones válidas")
                modelos_entrenados.append(None)
                continue
                
            # Ajustar modelo
            X = add_constant(datos[vars_modelo])
            y = datos['log_salario']
            modelo = sm.OLS(y, X).fit()
            modelos_entrenados.append(modelo)
            
            # Preparar resultados para tabla
            fila = {'Modelo': f"({i})", 'N': len(datos), 'R2': round(modelo.rsquared, 3)}
            
            # Agregar coeficientes con formato requerido
            for var in ['const'] + vars_modelo:
                coef = modelo.params[var]
                se = modelo.bse[var]
                p = modelo.pvalues[var]
                stars = '***' if p < 0.001 else '**' if p < 0.05 else '*' if p < 0.1 else ''
                fila[var] = f"{coef:.3f}{stars} ({se:.2f})"
            
            # Rellenar con NA las variables no usadas en este modelo
            todas_vars = ['edad', 'edad2', 'educ', 'mujer', 'tiene_cobertura', 'sabe_leer_escribir']
            for var in todas_vars:
                if var not in fila:
                    fila[var] = "NA"
            
            resultados.append(fila)
            
        except Exception as e:
            print(f"Error en modelo {i}: {str(e)}")
            modelos_entrenados.append(None)
            continue
    
    return resultados, modelos_entrenados

def generar_tabla_resultados(resultados):
    """Genera la tabla de resultados según formato solicitado"""
    columnas = ['Modelo', 'edad', 'edad2', 'educ', 'mujer',
               'tiene_cobertura', 'sabe_leer_escribir', 'N', 'R2']
    
    tabla_final = pd.DataFrame(resultados, columns=columnas)
    
    print("\n" + "="*80)
    print("Tabla 2. Estimación por regresión lineal de salarios usando la base de entrenamiento")
    print("Var. Dep: log(salario_semanal)")
    print("Nota: * p<0.1, ** p<0.05, *** p<0.001")
    print("="*80)
    
    return tabla_final

# -----------------------------------------------------------
# EJECUCIÓN PRINCIPAL
# -----------------------------------------------------------

# 1. Cargar datos (asumiendo que 'respondieron' o 'df_gba' está disponible)
if 'respondieron' in globals():
    df_principal = respondieron
elif 'df_gba' in globals():
    df_principal = df_gba
else:
    raise ValueError("No se encontró ni 'respondieron' ni 'df_gba' en los datos")

# 2. Filtrar solo ocupados (ESTADO == 1)
ocupados = df_principal[df_principal['ESTADO'] == 1].copy()

# 3. Dividir en train/test (70/30)
train_data = ocupados.sample(frac=0.7, random_state=444)
test_data = ocupados.drop(train_data.index)

# 4. Procesar datos
print("Preparando datos...")
ocupados_train = preparar_datos_para_modelos(train_data)
ocupados_test = preparar_datos_para_modelos(test_data)

# 5. Estimar modelos y generar tabla
print("Estimando modelos...")
resultados, modelos_entrenados = estimar_modelos(ocupados_train)
tabla_final = generar_tabla_resultados(resultados)

# Mostrar tabla
print(tabla_final.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Estadísticas descriptivas
print("\nEstadísticas descriptivas:")
print(f"- Muestra final: {len(ocupados_train)} ocupados en entrenamiento")
print(f"- Salario semanal promedio: ${ocupados_train['salario_semanal'].mean():,.2f}")
print(f"- Edad promedio: {ocupados_train['edad'].mean():.1f} años")
print(f"- Porcentaje de mujeres: {ocupados_train['mujer'].mean()*100:.1f}%")
print(f"- Porcentaje con cobertura médica: {ocupados_train['tiene_cobertura'].mean()*100:.1f}%")
print(f"- Personas que saben leer y escribir: {ocupados_train['sabe_leer_escribir'].mean()*100:.1f}%")

Preparando datos...
Estimando modelos...

Tabla 2. Estimación por regresión lineal de salarios usando la base de entrenamiento
Var. Dep: log(salario_semanal)
Nota: * p<0.1, ** p<0.05, *** p<0.001
Modelo            edad            edad2            educ           mujer tiene_cobertura sabe_leer_escribir    N    R2
   (1) 0.025*** (0.00)               NA              NA              NA              NA                 NA 3472 0.011
   (2) 0.086*** (0.02)  -0.001** (0.00)              NA              NA              NA                 NA 3472 0.013
   (3) 0.088*** (0.02)  -0.001** (0.00) 0.327*** (0.01)              NA              NA                 NA 3472 0.299
   (4) 0.068*** (0.02)  -0.000** (0.00) 0.250*** (0.01) 3.139*** (0.11)              NA                 NA 3472 0.437
   (5) 0.077*** (0.01) -0.001*** (0.00) 0.165*** (0.01) 1.758*** (0.09) 3.776*** (0.08)       0.265 (0.45) 3471 0.646

Estadísticas descriptivas:
- Muestra final: 3472 ocupados en entrenamiento
- Salario semanal pr

#### 3. Enfoque de Validación: Ahora para cada modelo estime el salario predicho de testeo (salario_semanal_test sombrerito) usando las observaciones separadas de testeo y los coeficientes estimados en el apartado anterior. Reporte y comente las siguientes métricas de testo para cada modelo:

##### Tabla 3. Performance por regresión lineal de la predicción de salarios usando la base de testeo Var. Dep: salario_semanal Modelo 1 (1) Modelo 2 (2) Modelo 3 (3) Modelo 4 (4) Modelo 5 (5) MSE test RMSE test MAE test

In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Asumiendo que add_constant es de statsmodels
from statsmodels.api import add_constant

# Define la función tal cual la tienes (o asegúrate de que esté definida en una celda anterior)
def evaluar_modelos_test_mejorado(modelos_entrenados, test_data_proc):
    """
    Evaluación mejorada con métricas adicionales y análisis de residuos
    """
    vars_modelos = [
        ['edad'],
        ['edad', 'edad2'],
        ['edad', 'edad2', 'educ'],
        ['edad', 'edad2', 'educ', 'mujer'],
        ['edad', 'edad2', 'educ', 'mujer', 'tiene_cobertura', 'sabe_leer_escribir']
    ]

    metricas = {
        'Modelo': [],
        'MSE (×1e12)': [],
        'RMSE': [],
        'MAE': [],
        'Error Relativo (%)': [],
        'R²_test': [],
        'N_test': []
    }

    for i, modelo in enumerate(modelos_entrenados, 1):
        try:
            vars_necesarias = vars_modelos[i-1]
            # Nos aseguramos de tener todas las columnas necesarias, incluyendo la variable dependiente,
            # antes de eliminar filas con NAs.
            cols_to_check = vars_necesarias + ['salario_semanal']
            # Filtramos solo las columnas que existen realmente en test_data_proc para evitar errores
            present_cols = [col for col in cols_to_check if col in test_data_proc.columns]
            datos_test = test_data_proc[present_cols].dropna()

            # Verificamos que aún haya datos válidos y la variable dependiente
            if 'salario_semanal' not in datos_test.columns or len(datos_test) == 0:
                 print(f"Saltando Modelo {i}: Variable dependiente 'salario_semanal' no encontrada o no hay datos válidos después de eliminar NAs.")
                 # Opcional: Añadir filas vacías o con error para mantener la estructura de la tabla
                 metricas['Modelo'].append(f'({i})')
                 metricas['MSE (×1e12)'].append('N/A')
                 metricas['RMSE'].append('N/A')
                 metricas['MAE'].append('N/A')
                 metricas['Error Relativo (%)'].append('N/A')
                 metricas['R²_test'].append('N/A')
                 metricas['N_test'].append(0)
                 continue

            # Usamos solo las variables predictoras que existen y están en la lista vars_necesarias
            X_test_vars = [v for v in vars_necesarias if v in datos_test.columns]
            if not X_test_vars:
                 print(f"Saltando Modelo {i}: No se encontraron variables predictoras válidas para el modelo {i} en los datos de test.")
                 metricas['Modelo'].append(f'({i})')
                 metricas['MSE (×1e12)'].append('N/A')
                 metricas['RMSE'].append('N/A')
                 metricas['MAE'].append('N/A')
                 metricas['Error Relativo (%)'].append('N/A')
                 metricas['R²_test'].append('N/A')
                 metricas['N_test'].append(len(datos_test)) # O 0, dependiendo de cómo quieras reportarlo
                 continue


            X_test = add_constant(datos_test[X_test_vars], has_constant='add') # Aseguramos que se añada la constante si no está
            y_test = datos_test['salario_semanal']
            y_pred = np.exp(modelo.predict(X_test)) # Predecimos en la escala original

            # Métricas básicas
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y_test, y_pred)

            # Métricas adicionales
            # Manejar división por cero si el promedio es 0
            error_relativo = (mae / y_test.mean()) * 100 if y_test.mean() != 0 else float('inf')

            # Calcular R2 en la escala original (Pseudo-R2 de test)
            # Esto mide qué tan bien las predicciones en la escala original explican la varianza de los valores reales en la escala original
            total_variance_test = np.var(y_test)
            # Manejar el caso si la varianza es cero (todos los valores de y_test son iguales)
            r2_test = 1 - (mse / total_variance_test) if total_variance_test != 0 else float('nan')


            metricas['Modelo'].append(f'({i})')
            metricas['MSE (×1e12)'].append(f"{mse/1e12:.2f}")
            metricas['RMSE'].append(f"${rmse:,.0f}") # Formato monetario con comas
            metricas['MAE'].append(f"${mae:,.0f}")   # Formato monetario con comas
            metricas['Error Relativo (%)'].append(f"{error_relativo:.1f}%")
            metricas['R²_test'].append(f"{r2_test:.3f}")
            metricas['N_test'].append(len(datos_test))

        except Exception as e:
            print(f"Error evaluando Modelo {i}: {str(e)}")
            # Añadir filas placeholder en caso de error para no romper la tabla
            metricas['Modelo'].append(f'({i})')
            metricas['MSE (×1e12)'].append('Error')
            metricas['RMSE'].append('Error')
            metricas['MAE'].append('Error')
            metricas['Error Relativo (%)'].append('Error')
            metricas['R²_test'].append('Error')
            metricas['N_test'].append('Error')
            continue

    return pd.DataFrame(metricas)
# Asegúrate de que las bibliotecas necesarias estén importadas
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Asumiendo que add_constant es de statsmodels
from statsmodels.api import add_constant

# Define la función tal cual la tienes (o asegúrate de que esté definida en una celda anterior)
def evaluar_modelos_test_mejorado(modelos_entrenados, test_data_proc):
    """
    Evaluación mejorada con métricas adicionales y análisis de residuos
    """
    vars_modelos = [
        ['edad'],
        ['edad', 'edad2'],
        ['edad', 'edad2', 'educ'],
        ['edad', 'edad2', 'educ', 'mujer'],
        ['edad', 'edad2', 'educ', 'mujer', 'tiene_cobertura', 'sabe_leer_escribir']
    ]

    metricas = {
        'Modelo': [],
        'MSE (×1e12)': [],
        'RMSE': [],
        'MAE': [],
        'Error Relativo (%)': [],
        'R²_test': [],
        'N_test': []
    }

    for i, modelo in enumerate(modelos_entrenados, 1):
        try:
            vars_necesarias = vars_modelos[i-1]
            # Nos aseguramos de tener todas las columnas necesarias, incluyendo la variable dependiente,
            # antes de eliminar filas con NAs.
            cols_to_check = vars_necesarias + ['salario_semanal']
            # Filtramos solo las columnas que existen realmente en test_data_proc para evitar errores
            present_cols = [col for col in cols_to_check if col in test_data_proc.columns]
            datos_test = test_data_proc[present_cols].dropna()

            # Verificamos que aún haya datos válidos y la variable dependiente
            if 'salario_semanal' not in datos_test.columns or len(datos_test) == 0:
                 print(f"Saltando Modelo {i}: Variable dependiente 'salario_semanal' no encontrada o no hay datos válidos después de eliminar NAs.")
                 # Opcional: Añadir filas vacías o con error para mantener la estructura de la tabla
                 metricas['Modelo'].append(f'({i})')
                 metricas['MSE (×1e12)'].append('N/A')
                 metricas['RMSE'].append('N/A')
                 metricas['MAE'].append('N/A')
                 metricas['Error Relativo (%)'].append('N/A')
                 metricas['R²_test'].append('N/A')
                 metricas['N_test'].append(0)
                 continue

            # Usamos solo las variables predictoras que existen y están en la lista vars_necesarias
            X_test_vars = [v for v in vars_necesarias if v in datos_test.columns]
            if not X_test_vars:
                 print(f"Saltando Modelo {i}: No se encontraron variables predictoras válidas para el modelo {i} en los datos de test.")
                 metricas['Modelo'].append(f'({i})')
                 metricas['MSE (×1e12)'].append('N/A')
                 metricas['RMSE'].append('N/A')
                 metricas['MAE'].append('N/A')
                 metricas['Error Relativo (%)'].append('N/A')
                 metricas['R²_test'].append('N/A')
                 metricas['N_test'].append(len(datos_test)) # O 0, dependiendo de cómo quieras reportarlo
                 continue


            X_test = add_constant(datos_test[X_test_vars], has_constant='add') # Aseguramos que se añada la constante si no está
            y_test = datos_test['salario_semanal']
            y_pred = np.exp(modelo.predict(X_test)) # Predecimos en la escala original

            # Métricas básicas
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y_test, y_pred)

            # Métricas adicionales
            # Manejar división por cero si el promedio es 0
            error_relativo = (mae / y_test.mean()) * 100 if y_test.mean() != 0 else float('inf')

            # Calcular R2 en la escala original (Pseudo-R2 de test)
            # Esto mide qué tan bien las predicciones en la escala original explican la varianza de los valores reales en la escala original
            total_variance_test = np.var(y_test)
            # Manejar el caso si la varianza es cero (todos los valores de y_test son iguales)
            r2_test = 1 - (mse / total_variance_test) if total_variance_test != 0 else float('nan')


            metricas['Modelo'].append(f'({i})')
            metricas['MSE (×1e12)'].append(f"{mse/1e12:.2f}")
            metricas['RMSE'].append(f"${rmse:,.0f}") # Formato monetario con comas
            metricas['MAE'].append(f"${mae:,.0f}")   # Formato monetario con comas
            metricas['Error Relativo (%)'].append(f"{error_relativo:.1f}%")
            metricas['R²_test'].append(f"{r2_test:.3f}")
            metricas['N_test'].append(len(datos_test))

        except Exception as e:
            print(f"Error evaluando Modelo {i}: {str(e)}")
            # Añadir filas placeholder en caso de error para no romper la tabla
            metricas['Modelo'].append(f'({i})')
            metricas['MSE (×1e12)'].append('Error')
            metricas['RMSE'].append('Error')
            metricas['MAE'].append('Error')
            metricas['Error Relativo (%)'].append('Error')
            metricas['R²_test'].append('Error')
            metricas['N_test'].append('Error')
            continue

    return pd.DataFrame(metricas)


# Asumiendo que 'modelos_entrenados' es la lista de modelos entrenados
# y 'ocupados_test' es el DataFrame con los datos de test procesados.
# Debes asegurarte de que estas variables existan antes de ejecutar esta celda.

# Uso de la función mejorada para obtener el DataFrame
tabla_performance_mejorada = evaluar_modelos_test_mejorado(modelos_entrenados, ocupados_test)

# Visualización en Jupyter Notebook:
# En lugar de usar print, simplemente ponemos el nombre de la variable DataFrame
# como la última línea de la celda. Jupyter lo renderizará automáticamente como una tabla.
tabla_performance_mejorada

,Modelo,MSE (×1e12),RMSE,MAE,Error Relativo (%),R²_test,N_test
0,(1),0.02,"$146,968","$79,341",100.1%,-0.354,1481
1,(2),0.02,"$146,903","$79,332",100.1%,-0.353,1481
2,(3),0.02,"$136,071","$74,318",93.7%,-0.160,1480
3,(4),0.02,"$128,174","$66,545",83.9%,-0.029,1480
4,(5),0.04,"$196,794","$96,515",121.7%,-1.426,1480
